# 分點籌碼量化：特徵工程 + 規則型/ML 回測（以 8271 為例）

本 notebook 會：
- 讀取 `<stock_id>.sqlite`（分點明細 + 日股價/還原股價）
- 產生 stock-day 特徵（topK 淨買佔比、HHI 集中度、價格/量能）
- 回測 1–5 日持有期：
  - 規則型 R1 / R2 baseline
  - ML：Ridge walk-forward（避免偷看）
- 輸出：features/trades/equity/metrics 到 `outputs/`


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def _find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    return start


repo_root = _find_repo_root(Path.cwd())
sys.path.insert(0, str(repo_root))

from research.broker_flow import (
    DEFAULT_BUY_COST,
    DEFAULT_SELL_COST,
    backtest_fixed_hold,
    build_day_features,
    compute_forward_return,
    load_broker_daily_agg,
    load_stock_price_table,
    make_rule_signals,
    resolve_stock_db_path,
    walk_forward_ridge_predict,
)

pd.set_option("display.max_columns", 200)
plt.style.use("seaborn-v0_8")

STOCK_ID = "8271"
START_DATE = "2023-03-05"
END_DATE = "2026-03-05"  # 若當日尚無資料，會自動以 DB 實際可用日期為準
HOLD_DAYS_LIST = [1, 2, 3, 4, 5]

TOP_K = 20
BUY_COST = DEFAULT_BUY_COST
SELL_COST = DEFAULT_SELL_COST

db_path = resolve_stock_db_path(STOCK_ID, repo_root)
if not db_path.exists():
    raise FileNotFoundError(f"DB not found: {db_path}")

outputs_dir = repo_root / "outputs" / STOCK_ID
outputs_dir.mkdir(parents=True, exist_ok=True)

db_mtime = int(db_path.stat().st_mtime)
cache_broker_daily = outputs_dir / f"{STOCK_ID}_broker_daily_top{TOP_K}_{START_DATE}_{END_DATE}_{db_mtime}.parquet"
features_out = outputs_dir / f"{STOCK_ID}_day_features_top{TOP_K}_{START_DATE}_{END_DATE}_{db_mtime}.parquet"
metrics_out = outputs_dir / f"{STOCK_ID}_metrics_top{TOP_K}_{START_DATE}_{END_DATE}_{db_mtime}.csv"

print("repo_root:", repo_root)
print("db_path:", db_path)
print("outputs_dir:", outputs_dir)


In [ ]:
# 1) Load price data
price_raw = load_stock_price_table(
    db_path=db_path,
    table_name="stock_price",
    stock_id=STOCK_ID,
    start_date=START_DATE,
    end_date=END_DATE,
)
price_adj = load_stock_price_table(
    db_path=db_path,
    table_name="stock_price_adj",
    stock_id=STOCK_ID,
    start_date=START_DATE,
    end_date=END_DATE,
)

if price_adj.empty:
    raise RuntimeError("stock_price_adj is empty; please backfill price data first.")

# Drop non-tradable rows (missing open/close)
price_adj = price_adj.dropna(subset=["open", "close"])  # columns are open/high/low/close after load
price_raw = price_raw.dropna(subset=["open", "close"]) if not price_raw.empty else price_raw

print("price_adj range:", price_adj.index.min().date(), "~", price_adj.index.max().date(), "rows:", len(price_adj))
if not price_raw.empty:
    print("price_raw range:", price_raw.index.min().date(), "~", price_raw.index.max().date(), "rows:", len(price_raw))

display(price_adj.head())


In [ ]:
# 2) Load broker daily aggregates (per broker per day)
broker_daily = load_broker_daily_agg(
    db_path=db_path,
    start_date=START_DATE,
    end_date=END_DATE,
    cache_path=cache_broker_daily,
    show_progress=True,
)

print("broker_daily rows:", len(broker_daily))
display(broker_daily.head())


In [ ]:
# 3) Build stock-day features
features = build_day_features(price_raw=price_raw, price_adj=price_adj, broker_daily=broker_daily, top_k=TOP_K)
features = features.replace([np.inf, -np.inf], np.nan)

# Keep only date range available in adj prices
features = features.loc[(features.index >= pd.to_datetime(START_DATE)) & (features.index <= pd.to_datetime(END_DATE))]
print("features rows:", len(features))

display(features[["open_adj", "close_adj", "volume_adj", "topk_net_share", "buy_hhi", "ret_5", "vol_20"]].tail())

features.to_parquet(features_out, index=True)
print("Saved:", features_out)


## 快速資料檢查（診斷）

- `net_consistency = abs(sum_net)/volume` 越接近 0 越合理
- `buy_sell_ratio = sum_buy/sum_sell` 越接近 1 越合理


In [ ]:
diag = features[["net_consistency", "buy_sell_ratio", "active_brokers", "topk_net_share", "buy_hhi", "sell_hhi"]].copy()
display(diag.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))

fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
features["close_adj"].plot(ax=ax[0], title=f"{STOCK_ID} close_adj")
features["topk_net_share"].plot(ax=ax[1], title=f"top{TOP_K}_net_share")
plt.tight_layout()
plt.show()


## 規則型策略回測（R1 / R2）

In [ ]:
rule_signals = make_rule_signals(features)
list(rule_signals.keys())


In [ ]:
rule_results: dict[str, dict[int, object]] = {}
rows = []

for strat_name, sig in rule_signals.items():
    rule_results[strat_name] = {}
    for h in HOLD_DAYS_LIST:
        res = backtest_fixed_hold(
            features,
            sig,
            hold_days=h,
            buy_cost=BUY_COST,
            sell_cost=SELL_COST,
            open_col="open_adj",
            close_col="close_adj",
        )
        rule_results[strat_name][h] = res
        rows.append({"strategy": strat_name, "hold_days": h, **res.metrics})

metrics_rule = pd.DataFrame(rows).sort_values(["strategy", "hold_days"]).reset_index(drop=True)
display(metrics_rule)


## ML 策略回測（Ridge + walk-forward）

In [ ]:
feature_cols = [
    "ret_1",
    "ret_3",
    "ret_5",
    "ret_20",
    "vol_5",
    "vol_20",
    "hl_range",
    "oc_return",
    "log_vol",
    "vol_z20",
    "topk_net_share",
    "topk_net_share_3d",
    "buy_hhi",
    "sell_hhi",
    "active_brokers",
    "net_std_share",
]

tradeable_ml = (
    (features["volume_adj"].fillna(0) > 0)
    & features["open_adj"].notna()
    & features["close_adj"].notna()
)

ml_rows = []
ml_results: dict[int, object] = {}

for h in HOLD_DAYS_LIST:
    y = compute_forward_return(features, hold_days=h, buy_cost=BUY_COST, sell_cost=SELL_COST)
    pred = walk_forward_ridge_predict(
        features,
        y,
        feature_cols=feature_cols,
        hold_days=h,
        train_window=504,
        min_train=252,
        alpha=1.0,
        show_progress=True,
    )
    sig_ml = tradeable_ml & pred.notna() & (pred > 0)
    res = backtest_fixed_hold(
        features,
        sig_ml,
        hold_days=h,
        buy_cost=BUY_COST,
        sell_cost=SELL_COST,
        open_col="open_adj",
        close_col="close_adj",
    )
    ml_results[h] = res
    ml_rows.append({"strategy": "ML_Ridge", "hold_days": h, **res.metrics})

metrics_ml = pd.DataFrame(ml_rows).sort_values(["hold_days"]).reset_index(drop=True)
display(metrics_ml)


## 匯總與輸出

- 輸出 metrics / trades / equity 到 `outputs/<stock_id>/`
- 繪製一條代表性的 equity curve（挑 Sharpe 最大的 hold_days）


In [ ]:
metrics_all = pd.concat([metrics_rule, metrics_ml], ignore_index=True)
metrics_all.to_csv(metrics_out, index=False, encoding="utf-8-sig")
print("Saved:", metrics_out)

# Save trades/equity
for strat_name, by_hold in rule_results.items():
    for h, res in by_hold.items():
        trades_path = outputs_dir / f"{STOCK_ID}_trades_{strat_name}_h{h}_{db_mtime}.csv"
        equity_path = outputs_dir / f"{STOCK_ID}_equity_{strat_name}_h{h}_{db_mtime}.csv"
        res.trades.to_csv(trades_path, index=False, encoding="utf-8-sig")
        res.equity.rename("equity").to_csv(equity_path, index=True)

for h, res in ml_results.items():
    trades_path = outputs_dir / f"{STOCK_ID}_trades_ML_Ridge_h{h}_{db_mtime}.csv"
    equity_path = outputs_dir / f"{STOCK_ID}_equity_ML_Ridge_h{h}_{db_mtime}.csv"
    res.trades.to_csv(trades_path, index=False, encoding="utf-8-sig")
    res.equity.rename("equity").to_csv(equity_path, index=True)

# Plot representative curves (best Sharpe per strategy)
def pick_best_sharpe(df: pd.DataFrame, strategy: str) -> int:
    sub = df[df["strategy"] == strategy].copy()
    sub = sub.replace([np.inf, -np.inf], np.nan)
    sub = sub.dropna(subset=["sharpe"])  # may be NaN if too few trades
    if sub.empty:
        return HOLD_DAYS_LIST[0]
    return int(sub.sort_values("sharpe", ascending=False).iloc[0]["hold_days"])

fig, ax = plt.subplots(1, 1, figsize=(12, 4))
for strat in ["R1_FlowBreakout", "R2_ConcentrationBreakout", "ML_Ridge"]:
    best_h = pick_best_sharpe(metrics_all, strat)
    if strat == "ML_Ridge":
        eq = ml_results[best_h].equity
    else:
        eq = rule_results[strat][best_h].equity
    eq.plot(ax=ax, label=f"{strat} (h={best_h})")

ax.set_title(f"{STOCK_ID} Equity (best Sharpe hold_days per strategy)")
ax.legend()
plt.tight_layout()
plt.show()

display(metrics_all.sort_values(["strategy", "hold_days"]))
